In [1]:
import pysftp
import sys
import os
import pandas as pd

c:\Users\jonas\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#download files again (yes/no)?
download = "yes"

#set year for data creation
year = '2024'

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [4]:
cnopts = pysftp.CnOpts()
cnopts.hostkeys = None

c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jonas\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


In [5]:
dir_out = "../parsed_data/"

In [6]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [8]:
with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
    print("Connection succesfully established.")

    # show list of files
    files = sftp.listdir('/TP_export/')   
    print(files)

Connection succesfully established.
['AcceptedAggregatedOffers_17.1.D', 'ActivatedBalancingEnergy_17.1.E', 'ActualCapacitiesAndOutlookOnFrequencyRestorationReserveAndReplacementReserve_SOGL_188.3_188.4_189.2_189.3_r3', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r2.1', 'ActualGenerationOutputPerGenerationUnit_16.1.A_r3', 'ActualTotalLoad_6.1.A', 'ActualTotalLoad_6.1.A_r3', 'AggregatedBalancingEnergyBids_12.3.E_r3', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D', 'AggregatedFillingRateOfWaterReservoirsAndHydroStoragePlants_16.1.D_r3', 'AggregatedGenerationPerType_16.1.B_C', 'AmountAndPricesPaidOfBalancingReservesUnderContract_17.1.B_C_r2', 'AmountOfBalancingReservesUnderContract_17.1.B', 'AuctionRevenue_12.1.A_r3', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r2', 'BalancingBorderCapacityLimitations_IFs_4.3_4.4_r3', 'ChangesInActualAvailabilityOfConsumptionUnits_7.1.B', 'ChangesInActualAvailabilityOfOffshoreGridInfrastructureReasons_10.1.C', 'ChangesInActual

## load data

In [9]:
#set paths and get file names
path_load = path+'ActualTotalLoad_6.1.A/'
path_load_local = path_local+'load/'
with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_load)
    #download files
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [10]:
#download aggregated load data (ActualTotalLoad)
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password,cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_load+file,path_load_local+file)
            print('Successfully downloaded file '+file)

Successfully downloaded file 2024_01_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_02_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_03_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_04_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_05_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_06_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_07_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_08_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_09_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_10_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_11_ActualTotalLoad_6.1.A.csv
Successfully downloaded file 2024_12_ActualTotalLoad_6.1.A.csv


In [12]:
df_temp = pd.read_csv(path_load_local+file,
                          decimal=".",sep="\t",
                          parse_dates=True, index_col="DateTime")

In [13]:
df_temp.head()

,ResolutionCode,AreaCode,AreaTypeCode,AreaName,MapCode,TotalLoadValue,UpdateTime
DateTime,,,,,,,
2024-01-27 00:00:00,PT60M,10YPL-AREA-----S,CTA,PL CTA,PL,17448.54,2024-01-27 06:31:40.040
2024-01-27 01:00:00,PT15M,10YRO-TEL------P,CTY,RO CTY,RO,5758.00,2024-01-27 02:31:31.031
2024-01-27 01:00:00,PT15M,10YHU-MAVIR----U,CTY,HU CTY,HU,4794.33,2024-01-27 02:31:37.037
2024-01-27 00:00:00,PT60M,10YCS-SERBIATSOV,BZN,RS BZN,RS,4511.00,2024-01-28 08:03:24.024
2024-01-27 00:45:00,PT15M,10YNL----------L,BZN,NL BZN,NL,11978.97,2024-10-07 20:46:52.052


In [14]:
#combine files to one data frame
df_load = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_load_local+file,
                          decimal=".",sep="\t",
                          parse_dates=True, index_col="DateTime")
    df_load = pd.concat([df_load,df_temp])
df_load = df_load[df_load.AreaTypeCode == "CTY"].drop(["AreaTypeCode",'ResolutionCode','AreaName','UpdateTime','AreaCode'], axis=1).reset_index()
df_load = df_load.sort_values(by=['DateTime'])
df_load.info()

<class 'pandas.core.frame.DataFrame'>
Index: 655352 entries, 2111 to 654452
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DateTime        655352 non-null  datetime64[ns]
 1   MapCode         655352 non-null  object        
 2   TotalLoadValue  655352 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 20.0+ MB


In [15]:
df_load = df_load.rename(columns={"MapCode": "country", 'DateTime':'date','TotalLoadValue':'load'})
df_load = df_load.pivot_table("load", "date", "country")

In [16]:
df_load.head()

country,AL,AT,BA,BE,BG,CH,CY,CZ,DE,DK,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,XK
date,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,731.0,5577.6,836.68,7335.32,3743.14,7692.42,414.94,5087.95,39336.85,3704.45,...,11311.90,18363.75,13695.19,5135.2,5046.0,4021.0,16763.0,891.70,2280.0,1132.36
2024-01-01 00:15:00,731.0,5511.2,NaN,7265.05,NaN,NaN,NaN,NaN,38991.10,NaN,...,11280.72,NaN,NaN,NaN,4986.0,NaN,NaN,NaN,NaN,NaN
2024-01-01 00:30:00,731.0,5443.6,NaN,7247.54,NaN,NaN,NaN,NaN,38615.81,NaN,...,11247.95,NaN,NaN,NaN,4947.0,NaN,NaN,NaN,NaN,NaN
2024-01-01 00:45:00,731.0,5389.6,NaN,7202.48,NaN,NaN,NaN,NaN,38328.84,NaN,...,11127.99,NaN,NaN,NaN,4916.0,NaN,NaN,NaN,NaN,NaN
2024-01-01 01:00:00,620.0,5423.6,815.14,7142.98,3608.08,7867.07,386.47,5005.89,38408.00,3638.33,...,11127.20,18224.37,13190.86,4962.5,4878.0,3863.0,16597.0,849.46,2252.0,1088.56


In [17]:
#some values are reported quarter hourly so we have to resample to hourly values
df_load_hourly = df_load.resample('1H').mean()

C:\Users\jonas\AppData\Local\Temp\ipykernel_22040\2118687946.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_load_hourly = df_load.resample('1H').mean()


Albania is completely missing so we copy profile from ME

In [18]:
df_load_hourly['AL'] = df_load_hourly['ME']*0.0001

In [19]:
#Lets see how complete values are
df_load_hourly.count().T

country
AL    8784
AT    8784
BA    8659
BE    8784
BG    8784
CH    8784
CY    2234
CZ    8784
DE    8784
DK    8784
EE    8782
ES    8784
FI    8784
FR    8784
GB    8551
GE    8347
GR    8783
HR    8784
HU    8784
IE    8717
IT    8784
LT    8784
LU    8784
LV    8784
MD    8609
ME    8784
MK    8400
NL    8784
NO    8784
PL    8784
PT    8784
RO    8784
RS    8784
SE    8784
SI    8743
SK    8764
XK    8778
dtype: int64

In [20]:
#Let's see how totals behave:
df_load_hourly.groupby(lambda x: x.year).sum().T/1000000
#they are too low so we prepare Eurostat yearly for upscaling in separate sheet

date,2024
country,
AL,0.000301
AT,58.802005
BA,9.591011
BE,80.972093
BG,36.767205
CH,59.628942
CY,1.189922
CZ,60.923177
DE,465.508457


In [21]:
df_load_hourly.to_csv(dir_out+'load_'+year+'_hourly_entsoe.csv', encoding="utf-8")